<a href="https://colab.research.google.com/github/ludtmit/baseball_pitch_prediction/blob/main/P1/baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P1: Data, split, and baseline results

Runs `P1/pipeline.py`: loads the shared 2026 Statcast snapshot, shows the chronological train/dev/test split, and scores three frequency baselines on **dev** with P4's `evaluate.py`.

The test split is not touched here. It is scored once, at the end, with `python P1/pipeline.py --split test`.

## Setup

In Colab this clones the repo. To skip the Statcast pull (several minutes), put `statcast_2026.csv.gz` from the team Drive into `data/` first.

In [1]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone -q https://github.com/ludtmit/baseball_pitch_prediction.git
    %cd baseball_pitch_prediction
    !pip install -q pybaseball

ROOT = Path.cwd() if (Path.cwd() / "P1").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / "P4")]

import pandas as pd
from P1.pipeline import CLASSES, DATA, baseline_probs, describe, load_snapshot, write_predictions
from evaluate import evaluate, load_predictions

## Data and split

Whole games in date order: earliest 70% train, next 15% dev, latest 15% test.

In [2]:
df = load_snapshot()
describe(df)
train, dev = df[df["split"] == "train"], df[df["split"] == "dev"]

train:  498,004 pitches, 1,700 games, 2026-03-25 to 2026-08-04
dev  :  107,233 pitches,   364 games, 2026-08-04 to 2026-08-31
test :  107,395 pitches,   365 games, 2026-08-31 to 2026-09-27


Class share (train): {'fastball': 0.553, 'slider': 0.222, 'offspeed': 0.145, 'breaking': 0.081}
dev pitches from pitchers unseen in train: 4.9%
test pitches from pitchers unseen in train: 9.1%


In [3]:
df[["game_date", "pitcher", "pitch_type", "pitch_cat", "count", "same_hand", "prev1", "prev2", "split"]].head(8)

,game_date,pitcher,pitch_type,pitch_cat,count,same_hand,prev1,prev2,split
0,2026-03-25,657277,SI,fastball,0-0,0,none,none,train
1,2026-03-25,657277,CH,offspeed,0-1,0,fastball,none,train
2,2026-03-25,657277,FF,fastball,0-2,0,offspeed,fastball,train
3,2026-03-25,657277,CH,offspeed,1-2,0,fastball,offspeed,train
4,2026-03-25,657277,CH,offspeed,1-2,0,offspeed,fastball,train
5,2026-03-25,657277,FF,fastball,1-2,0,offspeed,offspeed,train
6,2026-03-25,657277,SI,fastball,0-0,1,none,none,train
7,2026-03-25,657277,CH,offspeed,0-1,1,fastball,none,train


## Baselines on dev

All use training-set frequencies only:

- **overall_mix**: the league-wide mix, so the top pick is always fastball
- **pitcher_mix**: the pitcher's own mix, so the top pick is their most common category (the proposal baseline)
- **pitcher_count_mix**: the pitcher's mix in the current count

In [4]:
results = {}
for name, probs in baseline_probs(train, dev).items():
    path = DATA / f"dev_{name}.csv"
    write_predictions(dev, probs, path)
    results[name] = evaluate(load_predictions(path))

pd.DataFrame({name: {"accuracy": r["accuracy"], "log_loss": r["log_loss"]} for name, r in results.items()}).T.round(4)

,accuracy,log_loss
overall_mix,0.5597,1.1370
pitcher_mix,0.5659,1.0017
pitcher_count_mix,0.5707,0.9780


## Confusion matrix: proposal baseline

Rows are the actual category, columns the predicted one.

In [5]:
cm = pd.DataFrame(results["pitcher_mix"]["confusion_matrix"], index=CLASSES, columns=CLASSES)
cm["recall"] = (pd.Series(cm.to_numpy().diagonal(), index=CLASSES) / cm[list(CLASSES)].sum(axis=1)).round(3)
cm

,fastball,offspeed,breaking,slider,recall
fastball,57977,337,30,1669,0.966
offspeed,15264,365,0,263,0.023
breaking,7999,12,59,285,0.007
slider,20478,211,0,2284,0.099


## Takeaways

- Fastballs are 56% of pitches, so accuracy barely separates the baselines (56.0% → 56.6% → 57.1%).
- Log loss does separate them: pitcher mix 1.002, pitcher × count 0.978. The P3 general MLP (no player IDs) reached 1.043, which is **worse** than the pitcher baseline, so the model needs pitcher identity.
- The proposal baseline almost never predicts breaking balls (59 of 8,355 correct), which is why the model uses class-weighted loss.